In [1]:
%run functions.ipynb

import os

os.makedirs("../submission", exist_ok=True)

df = load_data()
df.head()

,yt_true
date,
1946-01-01,890
1946-02-01,992
1946-03-01,979
1946-04-01,959
1946-05-01,1110


In [2]:
# Model 8: seasonal ARIMA(1,1,1)(0,1,1)_12 fitted on the training period
import warnings

from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")

y_train = df.yt_true.iloc[:-24].reset_index(drop=True)

sarima = SARIMAX(
    y_train,
    order=(1, 1, 1),
    seasonal_order=(0, 1, 1, 12),
).fit(disp=False)
sarima.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                     SARIMAX Results                                      
==========================================================================================
Dep. Variable:                            yt_true   No. Observations:                  228
Model:             SARIMAX(1, 1, 1)x(0, 1, 1, 12)   Log Likelihood               -1482.657
Date:                            Fri, 25 Sep 2026   AIC                           2973.314
Time:                                    14:10:33   BIC                           2986.796
Sample:                                         0   HQIC                          2978.761
                                            - 228                                         
Covariance Type:                              opg                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.1636      0.116      1.413      0.158      -0.063       0.390
ma.L1         -0.6887      0.092     -7.524      0.000      -0.868      -0.509
ma.S.L12      -0.7735      0.044    -17.554      0.000      -0.860      -0.687
sigma2      5.482e+04   4000.999     13.702      0.000     4.7e+04    6.27e+04
===================================================================================
Ljung-Box (L1) (Q):                   0.01   Jarque-Bera (JB):                58.24
Prob(Q):                              0.92   Prob(JB):                         0.00
Heteroskedasticity (H):               1.11   Skew:                             0.42
Prob(H) (two-sided):                  0.67   Kurtosis:                         5.40
===================================================================================

Warnings:
[1] Covariance matrix calculated using the outer product of gradients (complex-step).
"""

In [3]:
# In-sample one-step-ahead predictions + 24-month multi-step forecast
in_sample = sarima.predict(start=0, end=len(y_train) - 1)
forecast = sarima.forecast(steps=24)

df["yt_pred_sarima"] = np.concatenate([in_sample.values, forecast.values])
# The first 13 points are dominated by the differencing initialization
df.iloc[:13, df.columns.get_loc("yt_pred_sarima")] = np.nan
plot_time_series(df[["yt_true", "yt_pred_sarima"]])

<Figure size 1200x400 with 1 Axes>

In [4]:
metrics = compute_evaluation_metrics(df[["yt_true", "yt_pred_sarima"]].dropna())
save_forecasts(df)
save_metrics(metrics)
metrics

,Metrics,yt_pred_sarima
0,MSE Train,56839.88
1,MSE Test,140704.08
2,MAE Train,174.22
3,MAE Test,304.84
